# Calculate the effective MIZ width of the AltiKa dataset

## Get datastore

In [ ]:
from intake import cat
from xarray import DataTree, map_over_datasets
from dask.distributed import Client
import glob
import xarray as xr
import numpy as np
from datetime import timedelta
import cf_xarray as cfxr
import xesmf
import xesmf as xe
import re
import os
import time
import intake
from tqdm.notebook import tqdm

import cftime
import calendar
import cftime
import calendar
import imageio.v2 as imageio
from pathlib import Path

# Plotting
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmocean.cm as cmo
import matplotlib.lines as mlines
import cartopy.feature as cft

# Import my functions
functions_path = os.path.abspath("/home/566/nd0349/access-om3-analysis/functions")
if functions_path not in sys.path:
    sys.path.append(functions_path)
from get_files import *
from plot_settings import *
from fstd import *
from parameters import *
π = np.pi
test()

In [ ]:
datastore_name = "/scratch/ps29/nd0349/access-om3/archive/WW3-standalone-ERA5-dice-2013/experiment_datastore.json"
# datastore_name = "/scratch/tm70/ek4684/access-om3/archive/Standalone_WW3_ERA5_dice/experiment_datastore.json"


model = Path(datastore_name).parts[-4] # "MCW-ERA5" # 
experiment = Path(datastore_name).parts[-2]
iaf = "ryf" not in experiment

output_frequency = "fx" # "1day" "1mon"
print(model, experiment)

In [ ]:
# experiment = experiment + "_ezhil"

In [ ]:
figpath = f"/g/data/ps29/nd0349/access-om3-analysis-figs/{experiment}"
os.makedirs(figpath, exist_ok=True)

In [ ]:
client = Client(threads_per_worker=1)
client

In [ ]:
print(client.dashboard_link)

## See what data is available

In [ ]:
datastore = intake.open_esm_datastore(
    f"{datastore_name}", 
    columns_with_iterables=[
            "variable",
            "variable_long_name",
            "variable_standard_name",
            "variable_cell_methods",
            "variable_units",
    ] # This is important
)

datastore

In [ ]:
datastore.unique().frequency
datastore.search(frequency=output_frequency).unique().variable#[0:10]
# esm_datastore.df.frequency.unique()

In [ ]:
def available_variables(datastore):
    """Return a pandas dataframe summarising the variables in a datastore"""
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

In [ ]:
datastore.unique()['variable']

In [ ]:
datastore_filtered = datastore.search(
    variable=["aice_m", "hi_m", "fsdrad_m", "wave_sig_ht_m", "uvel_m", "vvel_m", "uatm_m", "vatm_m"], #frequency=output_frequency, #require_all_on="path"
)
datastore_filtered.unique()
# access-om3.cice.1mon.mean.1984-01.nc

In [ ]:
# datastore_filtered = datastore.search(
#     variable=["TLON", "TLAT"], #frequency=output_frequency, #require_all_on="path"
# )
# datastore_filtered.unique()

## Load in WW3 data

In [ ]:
xarray_open_kwargs = {"chunks": {"time": 12, "nx": -1, "ny": -1}}
ds_ww3 = datastore.search(variable=["EF", "HS", "ICE", "ICEF", "ICEH", "THM", "FP0", "UAX", "UAY"], require_all_on="path").to_dask(xarray_open_kwargs=xarray_open_kwargs)
files = datastore.search(variable=["EF", "HS", "ICE", "ICEF", "ICEH", "THM", "FP0", "UAX", "UAY"], require_all_on="path").unique().filename

import re
clean_dates = [re.search(r"\d{4}-\d{2}-\d{2}", f).group() for f in files]
tmp_dates = pd.to_datetime(clean_dates)

grid_ds = xr.open_dataset('/g/data/vk83/configurations/inputs/access-om3/cice/grids/global.1deg/2024.05.14/grid.nc')
ds_ww3.coords['TLON'] = np.degrees(grid_ds['tlon'])
ds_ww3.coords['TLAT'] = np.degrees(grid_ds['tlat'])
ds_ww3['tarea'] = grid_ds['tarea']
ds_ww3['HTE'] = grid_ds['hte']/100 # cm to m
ds_ww3 = ds_ww3.rename({"ni": "nx", "nj": "ny"})

# coords = datastore.search(variable=["geolat", "geolon"]).to_dask().compute()
# coords = coords.fillna(0.0)
# ds_ww3 = ds_ww3.assign_coords(coords)
ds_ww3

## Compare with Fraser et al. (2026)

In [ ]:
def ReadInAltika(version, year=2019):
    if version == '0.6':
        month_range = range(1,13)
        df = pd.concat((pd.read_csv('/g/data/ps29/nd0349/Fraser-2024/data/v0_15/' + str(year) + ("%02d" % (month,)) + '_output_v0_6.csv') 
                        for month in tqdm(month_range, total = len(month_range), desc = "Reading in Alex's data")),
                        ignore_index=True)
        
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
        
    
    elif version == '0.10':
        # Version 0.10
        df_raw = pd.read_csv('data/v0_10/' + str(year) + '_all_output_v0_10.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
    elif version == '0.11':
        # Version 0.11
        df_raw = pd.read_csv('data/v0_11/' + str(year) + '_all_output_v0_11.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
    elif version == '0.12':
        # Version 0.12
        df_raw = pd.read_csv('data/v0_12/' + str(year) + '_all_output_v0_12.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
        
    elif version == '0.15':
        # Version 0.15
        df_raw = pd.read_csv('/g/data/ps29/nd0349/Fraser-2024/data/v0_15/' + str(year) + '_all_output_v0_15.csv')
        row_temp = df_raw.loc[0,:]
        row_temp.values
        # Fill the first row with temporary data
        df = pd.DataFrame([row_temp], columns = df_raw.columns)
        
        numberRows,NumberCols = df_raw.shape
        
        for i in range(numberRows):
            row_temp = df_raw.loc[i,:]
            row = pd.DataFrame([row_temp], columns = df_raw.columns)
            if  (row['too_many_switches_flag'].values == 0) & (row['hit_continent_flag'].values == 0) & (row['ice_edge_diff_flag'].values == 0) & (row['latAtInnerMIZ'].values < row['latAtAltiKaEdge'].values) & (row['latAtInnerMIZ'].values < row['latAtMyEdge'].values) & (np.abs(row['lonAtInnerMIZ'].values - row['lonAtAltiKaEdge'].values) < 10):
                df = pd.concat([df,row])
        df.drop([0])
        
    # Add dates to dataframe
    df['date'] = pd.to_datetime(df["first_meas_time"])#, format='%Y-%m-%d').dt.round("d")
    df['day'] = pd.to_datetime(df['date']).dt.day
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month
    return df

In [ ]:
from tqdm.notebook import tqdm

years = range(2013, 2024)
dfs = []

# swh_min = 10**-12
# swh_max = 100
# miz_max = 10000
# miz_min = -10**-12

for year in tqdm(years):
    # print(year)
    df = pd.DataFrame(ReadInAltika(version='0.15', year=year))
    df = df[['first_meas_time', 'swhAtMyEdge', 'lonAtMyEdge', 'latAtMyEdge', 'lonAtAltiKaEdge','latAtAltiKaEdge', 'lonAtInnerMIZ', 'latAtInnerMIZ', 'mizWidthAlongTrackFromMyEdge', 'mizWidthAlongTrackFromAltikaEdge']]
    df['date'] = pd.to_datetime(df["first_meas_time"]).dt.date # , format='%Y-%m-%d %H:%M:%S.%f').dt.date
    df['day'] = pd.to_datetime(df['date']).dt.day
    df['year'] = pd.to_datetime(df['date']).dt.year
    df['month'] = pd.to_datetime(df['date']).dt.month
    df['mizwidth_lat'] = abs(df['latAtAltiKaEdge'] - df['latAtInnerMIZ'])*111.32 # Alex's conversion from latitudes to km

    dfs.append(df)
# Combine into one dataframe
df_all = pd.concat(dfs, ignore_index=True)
# df_tmp = df_all.copy()
# condition_met_df = (df_tmp['swhAtMyEdge'] < swh_max) & (df_tmp['swhAtMyEdge'] > swh_min) & (df_tmp['mizWidthAlongTrackFromAltikaEdge'] < miz_max) & (df_tmp['mizWidthAlongTrackFromAltikaEdge'] > miz_min)
# df_all = df_tmp.where(condition_met_df)
df_all.head()

In [ ]:
month = 9
year = 2019

In [ ]:
df_all["time"] = pd.to_datetime(df_all["first_meas_time"])
df_month = df_all[
    (df_all["time"].dt.year == year) &
    (df_all["time"].dt.month == month)
].copy()
df_month.head()

In [ ]:
df_example = df_month.iloc[0].copy()
plt.scatter(df_example['lonAtInnerMIZ'], df_example['latAtInnerMIZ'])
plt.scatter(df_example['lonAtAltiKaEdge'], df_example['latAtAltiKaEdge'])
plt.scatter(df_example['lonAtMyEdge'], df_example['latAtMyEdge'])

In [ ]:
def haversine_km(lon1, lat1, lon2, lat2, radius=6371.0):
    lon1 = np.deg2rad(lon1)
    lat1 = np.deg2rad(lat1)
    lon2 = np.deg2rad(lon2)
    lat2 = np.deg2rad(lat2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))
    return radius * c


In [ ]:
df_example = df_month.iloc[0].copy()

dist_altika_to_inner = haversine_km(
    df_example["lonAtAltiKaEdge"],
    df_example["latAtAltiKaEdge"],
    df_example["lonAtInnerMIZ"],
    df_example["latAtInnerMIZ"]
)

dist_myedge_to_inner = haversine_km(
    df_example["lonAtMyEdge"],
    df_example["latAtMyEdge"],
    df_example["lonAtInnerMIZ"],
    df_example["latAtInnerMIZ"]
)

dist_myedge_to_altika = haversine_km(
    df_example["lonAtMyEdge"],
    df_example["latAtMyEdge"],
    df_example["lonAtAltiKaEdge"],
    df_example["latAtAltiKaEdge"]
)

print("AltiKa edge to inner MIZ:", dist_altika_to_inner, "km")
print("MyEdge to inner MIZ:", dist_myedge_to_inner, "km")
print("MyEdge to AltiKa edge:", dist_myedge_to_altika, "km")
print("AltiKa MIZ:", df_example["mizWidthAlongTrackFromAltikaEdge"], "km")
print("MyEdge MIZ:", df_example["mizWidthAlongTrackFromMyEdge"], "km")


In [ ]:
df_month["altika_myedge_dist_km"] = haversine_km(
    df_month["lonAtAltiKaEdge"],
    df_month["latAtAltiKaEdge"],
    df_month["lonAtMyEdge"],
    df_month["latAtMyEdge"]
)

df_month_filtered = df_month[df_month["altika_myedge_dist_km"] < 50].copy()
print(len(df_month), len(df_month_filtered))
df_month_filtered = df_month_filtered[df_month_filtered["mizWidthAlongTrackFromAltikaEdge"] < 500].copy()
print(len(df_month), len(df_month_filtered))

In [ ]:
df_month_sample = df_month_filtered.iloc[0:5]

### Read in ERA5 file

In [ ]:
month = 9
year = 2019

era5_files = sorted(glob(
    f"/g/data/rt52/era5/single-levels/reanalysis/ci/{year}/ci_era5_oper_sfc_{year}{month:02d}*.nc"
))

print(era5_files)

era5_file = era5_files[0]

ds_era5 = xr.open_dataset(era5_file, engine="netcdf4")
ds_era5

In [ ]:
# Example: daily or monthly mean ERA5 sea ice concentration
era5_ice_plot = ds_era5["siconc"].isel(time=0)

fig, ax = plt.subplots(
    figsize=(9, 7),
    subplot_kw={"projection": ccrs.SouthPolarStereo()}
)

era5_ice_plot.plot(
    ax=ax,
    x="longitude",
    y="latitude",
    transform=ccrs.PlateCarree(),
    cmap=cmo.ice,
    vmin=0,
    vmax=1,
    add_colorbar=True
)

ax.coastlines()
ax.set_extent([-180, 180, -90, -50], crs=ccrs.PlateCarree())

# Plot filtered df tracks
for _, row in df_month_sample.iterrows():
    ax.plot(
        [row["lonAtAltiKaEdge"], row["lonAtInnerMIZ"]],
        [row["latAtAltiKaEdge"], row["latAtInnerMIZ"]],
        color="yellow",
        linewidth=0.8,
        alpha=0.8,
        transform=ccrs.PlateCarree()
    )

# Optional: plot edge points
ax.scatter(
    df_month_sample["lonAtAltiKaEdge"],
    df_month_sample["latAtAltiKaEdge"],
    s=8,
    color="k",
    transform=ccrs.PlateCarree(),
    label="AltiKa edge"
)

ax.scatter(
    df_month_sample["lonAtInnerMIZ"],
    df_month_sample["latAtInnerMIZ"],
    s=8,
    color="red",
    transform=ccrs.PlateCarree(),
    label="Inner MIZ"
)

ax.legend()
plt.show()

## Calculate the effective MIZ width

In [ ]:
from scipy.interpolate import RegularGridInterpolator
import numpy as np

# Monthly mean ERA5 sea ice concentration
siconc_month = ds_era5["siconc"].mean("time")

# ERA5 grid
era_lats = siconc_month["latitude"].values
era_lons = siconc_month["longitude"].values
era_vals = siconc_month.values

# RegularGridInterpolator wants coordinates increasing
if era_lats[0] > era_lats[-1]:
    era_lats = era_lats[::-1]
    era_vals = era_vals[::-1, :]

siconc_interp = RegularGridInterpolator(
    (era_lats, era_lons),
    era_vals,
    bounds_error=False,
    fill_value=np.nan
)

def lon_to_180(lon):
    return ((lon + 180) % 360) - 180

def mean_siconc_along_track(row, start="altika", n_points=50):
    if start == "altika":
        lon0 = row["lonAtAltiKaEdge"]
        lat0 = row["latAtAltiKaEdge"]
    elif start == "myedge":
        lon0 = row["lonAtMyEdge"]
        lat0 = row["latAtMyEdge"]
    else:
        raise ValueError("start must be 'altika' or 'myedge'")

    lon1 = row["lonAtInnerMIZ"]
    lat1 = row["latAtInnerMIZ"]

    lon0 = lon_to_180(lon0)
    lon1 = lon_to_180(lon1)

    # Handle dateline crossing
    if abs(lon1 - lon0) > 180:
        if lon0 > lon1:
            lon1 += 360
        else:
            lon0 += 360

    track_lons = np.linspace(lon0, lon1, n_points)
    track_lats = np.linspace(lat0, lat1, n_points)

    track_lons = lon_to_180(track_lons)

    points = np.column_stack([track_lats, track_lons])
    siconc_track = siconc_interp(points)

    return np.nanmean(siconc_track)


In [ ]:
df_month_filtered = df_month_filtered.copy()
df_month_sample = df_month_filtered#.iloc[0:5]
df_month_sample

In [ ]:


df_month_filtered["mean_siconc_altika_track"] = df_month_filtered.apply(
    mean_siconc_along_track,
    axis=1,
    start="altika"
)

df_month_filtered["mean_siconc_myedge_track"] = df_month_filtered.apply(
    mean_siconc_along_track,
    axis=1,
    start="myedge"
)

df_month_filtered["mizWidthAlongTrackFromAltikaEdge_eff_era5"] = (
    df_month_filtered["mizWidthAlongTrackFromAltikaEdge"] *
    df_month_filtered["mean_siconc_altika_track"]
)

df_month_filtered["mizWidthAlongTrackFromMyEdge_eff_era5"] = (
    df_month_filtered["mizWidthAlongTrackFromMyEdge"] *
    df_month_filtered["mean_siconc_myedge_track"]
)


In [ ]:
df_month_filtered[
    [
        "mizWidthAlongTrackFromAltikaEdge",
        "mean_siconc_altika_track",
        "mizWidthAlongTrackFromAltikaEdge_eff_era5",
    ]
].head()


In [ ]:
df_month_filtered

## Automated

In [ ]:
from glob import glob
from scipy.interpolate import RegularGridInterpolator
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

def lon_to_180(lon):
    return ((lon + 180) % 360) - 180


def open_era5_siconc_month(year, month):
    files = sorted(glob(
        f"/g/data/rt52/era5/single-levels/reanalysis/ci/{year}/ci_era5_oper_sfc_{year}{month:02d}*.nc"
    ))

    if len(files) == 0:
        raise FileNotFoundError(f"No ERA5 siconc file found for {year}-{month:02d}")

    return xr.open_dataset(files[0], engine="netcdf4")


def make_siconc_interpolator_from_da(siconc_da):
    era_lats = siconc_da["latitude"].values
    era_lons = siconc_da["longitude"].values
    era_vals = siconc_da.values

    # RegularGridInterpolator requires increasing coordinate order
    if era_lats[0] > era_lats[-1]:
        era_lats = era_lats[::-1]
        era_vals = era_vals[::-1, :]

    return RegularGridInterpolator(
        (era_lats, era_lons),
        era_vals,
        bounds_error=False,
        fill_value=np.nan
    )


def mean_siconc_along_track(row, siconc_interp, start="altika", n_points=50):
    if start == "altika":
        lon0 = row["lonAtAltiKaEdge"]
        lat0 = row["latAtAltiKaEdge"]
    elif start == "myedge":
        lon0 = row["lonAtMyEdge"]
        lat0 = row["latAtMyEdge"]
    else:
        raise ValueError("start must be 'altika' or 'myedge'")

    lon1 = row["lonAtInnerMIZ"]
    lat1 = row["latAtInnerMIZ"]

    lon0 = lon_to_180(lon0)
    lon1 = lon_to_180(lon1)

    # Handle dateline crossing
    if abs(lon1 - lon0) > 180:
        if lon0 > lon1:
            lon1 += 360
        else:
            lon0 += 360

    track_lons = np.linspace(lon0, lon1, n_points)
    track_lats = np.linspace(lat0, lat1, n_points)

    track_lons = lon_to_180(track_lons)

    points = np.column_stack([track_lats, track_lons])
    siconc_track = siconc_interp(points)

    return np.nanmean(siconc_track)


def add_era5_effective_miz_for_month(df_all, year, month, n_points=50):
    ds_era5 = open_era5_siconc_month(year, month)

    df_month = df_all[
        (df_all["first_meas_time"].dt.year == year) &
        (df_all["first_meas_time"].dt.month == month)
    ].copy()

    if len(df_month) == 0:
        ds_era5.close()
        return df_month

    # Cleaning step 1: remove rows where AltiKa edge and MyEdge differ by >= 50 km
    df_month["altika_myedge_dist_km"] = haversine_km(
        df_month["lonAtAltiKaEdge"],
        df_month["latAtAltiKaEdge"],
        df_month["lonAtMyEdge"],
        df_month["latAtMyEdge"]
    )

    df_month = df_month[df_month["altika_myedge_dist_km"] < 50].copy()

    # Cleaning step 2: remove very large AltiKa MIZ widths
    df_month = df_month[df_month["mizWidthAlongTrackFromAltikaEdge"] < 500].copy()

    if len(df_month) == 0:
        ds_era5.close()
        return df_month

    # Match each satellite row to nearest ERA5 timestep
    era_times = pd.to_datetime(ds_era5["time"].values)

    nearest_idx = era_times.get_indexer(
        pd.to_datetime(df_month["first_meas_time"]),
        method="nearest"
    )

    df_month["era5_time"] = era_times[nearest_idx]
    df_month["era5_time_idx"] = nearest_idx

    mean_altika = np.full(len(df_month), np.nan)
    mean_myedge = np.full(len(df_month), np.nan)

    # Build one interpolator per ERA5 time used, then apply to rows at that time
    for time_idx, row_indices in tqdm(
        df_month.groupby("era5_time_idx").groups.items(),
        desc=f"ERA5 siconc {year}-{month:02d}"
    ):
        siconc_da = ds_era5["siconc"].isel(time=time_idx)
        siconc_interp = make_siconc_interpolator_from_da(siconc_da)

        for df_idx in row_indices:
            row = df_month.loc[df_idx]

            pos = df_month.index.get_loc(df_idx)

            mean_altika[pos] = mean_siconc_along_track(
                row,
                siconc_interp=siconc_interp,
                start="altika",
                n_points=n_points
            )

            mean_myedge[pos] = mean_siconc_along_track(
                row,
                siconc_interp=siconc_interp,
                start="myedge",
                n_points=n_points
            )

    df_month["mean_siconc_altika_track"] = mean_altika
    df_month["mean_siconc_myedge_track"] = mean_myedge

    df_month["mizWidthAlongTrackFromAltikaEdge_eff_era5"] = (
        df_month["mizWidthAlongTrackFromAltikaEdge"] *
        df_month["mean_siconc_altika_track"]
    )

    df_month["mizWidthAlongTrackFromMyEdge_eff_era5"] = (
        df_month["mizWidthAlongTrackFromMyEdge"] *
        df_month["mean_siconc_myedge_track"]
    )

    ds_era5.close()
    return df_month


In [ ]:
year = 2013
n_points = 50
start = f"{year}-01"
end = f"{year}-12"

df_all["first_meas_time"] = pd.to_datetime(df_all["first_meas_time"])
monthly_results = []

for period in tqdm(pd.period_range(start=start, end=end, freq="M")):
    try:
        df_month_eff = add_era5_effective_miz_for_month(
            df_all,
            year=period.year,
            month=period.month,
            n_points=n_points
        )
        monthly_results.append(df_month_eff)

    except FileNotFoundError as e:
        print(e)

df_all_eff = pd.concat(monthly_results, ignore_index=True)
out_file = f"/g/data/ps29/nd0349/Fraser-2024/data/cleaned/{year}_effective_hourly_{n_points}_v0_15.csv"
df_all_eff.to_csv(out_file, index=False)

In [ ]:
client.close()